# Custom variations and measurements

Define a joint variation and a measurement function. Functions used by workers must
live at module level. Also runnable as `python examples/07_custom.py`.
Jupyter uses one worker because functions defined in cells cannot be imported by spawned workers.

In [1]:

import sys
from pathlib import Path

import numpy as np

examples_dir = next(
    (root / "examples" for root in (Path.cwd(), *Path.cwd().parents)
     if (root / "examples" / "rc_study.py").is_file()),
    None,
)
if examples_dir is not None:
    sys.path.insert(0, str(examples_dir))

In [2]:
from rc_study import circuit, config  # noqa: E402

from spicefault import Experiment, Measurement  # noqa: E402
from spicefault.faults import ParametricFault  # noqa: E402
from spicefault.variation import Draw, JointVariation, ToleranceVariation  # noqa: E402


def matched_resistors(rng, netlist):
    """R1 and R2 from one reel: they share most of their deviation and differ a little.

    The type of the reel is drawn first and recorded as a label of the sample.
    """
    grade = str(rng.choice(["standard", "precision"]))
    spread = {"standard": 0.05, "precision": 0.005}[grade]
    common = 1.0 + spread * rng.uniform(-1, 1)
    values = {
        (name, "value"): netlist.value(name) * common * (1.0 + 0.001 * rng.uniform(-1, 1))
        for name in ("R1", "R2")
    }
    return Draw(values, {"resistor_grade": grade})

## A custom frequency-response measurement

In [3]:
def corner_frequency(plot):
    """Frequency at which the response has fallen 3 dB from its low-frequency value [Hz]."""
    frequency = plot["frequency"].real
    gain = np.abs(plot["v(out)"])
    below = np.flatnonzero(gain < gain[0] / np.sqrt(2.0))
    return float(frequency[below[0]]) if len(below) else float(frequency[-1])

In [4]:
if __name__ == "__main__":
    worker_count = 1 if "ipykernel" in sys.modules else 2
    experiment = Experiment(
        circuit,
        config=config,
        faults=[ParametricFault("C1", deviation=0.5)],
        variations=[
            JointVariation("resistors", (("R1", "value"), ("R2", "value")), matched_resistors),
            ToleranceVariation("C1", 0.05),
        ],
        measurements=[
            Measurement.value("v(out)", name="dc"),
            Measurement.custom("corner", corner_frequency, analysis="ac"),
        ],
        samples=6,
        seed=3,
    )
    table = experiment.run(workers=worker_count).to_frame()
    columns = ["fault_id", "resistor_grade", "p_R1_value", "p_R2_value", "dc", "corner"]
    print(table[columns].round(3).to_string(index=False))
    # matched resistors keep the divider ratio, whatever their grade
    print("\nspread of the DC gain:", f"{table['dc'].std():.5f}")
    print("recorded in the definition of the experiment:")
    print(" ", experiment.metadata()["variations"][0])

          fault_id resistor_grade  p_R1_value  p_R2_value  dc  corner
           healthy      precision    9958.335    9960.968 0.5 354.813
           healthy      precision   10042.879   10057.630 0.5 316.228
           healthy       standard    9725.772    9723.318 0.5 354.813
           healthy       standard   10432.780   10437.612 0.5 316.228
           healthy      precision    9982.086    9970.559 0.5 354.813
           healthy      precision    9992.148   10004.116 0.5 354.813
C1:parametric:+0.5      precision   10012.123   10001.625 0.5 223.872
C1:parametric:+0.5      precision    9987.781    9990.922 0.5 223.872
C1:parametric:+0.5       standard    9914.958    9915.367 0.5 223.872
C1:parametric:+0.5      precision    9976.931    9976.468 0.5 223.872
C1:parametric:+0.5       standard    9503.059    9497.282 0.5 251.189
C1:parametric:+0.5       standard   10004.005    9992.101 0.5 223.872

spread of the DC gain: 0.00022
recorded in the definition of the experiment:
  {'type': '